In [0]:
%run ../utilities/postgres_unload_util



In [0]:
%run ../utilities/file_utilities


In [0]:
dbutils.widgets.text("table_name","")

In [0]:
import json
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *


spark = SparkSession.builder.appName("benefit_modernization_apg_2_databricks_ingestion").getOrCreate()




 
try:
    # Load Postgres connection parameters from JSON file
    with open('../parameter_config/unload_apg_metdata.json', 'r') as f:
        conn_params = json.load(f)
except json.JSONDecodeError as e:
        print(f"Error parsing parameter config JSON: {e}")
        raise
except FileNotFoundError:
        print(" parameter config File not found! Check your path.")
        raise
except Exception as e:
        print(f"Unexpected error: {e}")
        raise

retry_val=4
source_schema_name=conn_params['source_schema_name']
table_name=dbutils.widgets.get("table_name")
conn_config=conn_params['conn_config']
source_database_name=conn_params['source_database_name']
target_catalog=conn_params['databricks_catalog_name']
checkpoint_file_path="/Volumes/benefit_modernization_dev/silver/delta_checkpoint_files"
delta_checkpoint_file=f"{checkpoint_file_path}/{table_name}.checkpoint"


#for table_name in table_name_vec:
 #       try:
  #              print(f"Loading table {table_name} from schema {source_schema_name}")
   #             df=load_postgres_table(conn_config,retry_val,source_schema_name,table_name,source_database_name)
    #            df_col=df.withColumn('ingest_ts', current_timestamp() )
     #           time.sleep(20)
      #          df.write.format("delta").mode("append").saveAsTable(F"{target_catalog}.bronze.{table_name}")
       #                                                                
        #except Exception as e:
         #       print(f"Error loading table {table_name} from schema {source_schema_name}")
          #      raise


if(path_exists(delta_checkpoint_file)==True):
        checkpoint_val=spark.read.csv(delta_checkpoint_file).take(1)[0][0];
        df=load_postgres_table(conn_config,retry_val,source_schema_name,table_name,source_database_name)
        df.filter(col("created_at") > to_timestamp(lit(checkpoint_val))).write.format("delta").mode("append").insertInto(F"{target_catalog}.bronze.{table_name}")
        time.sleep(20)
else:
        df=load_postgres_table(conn_config,retry_val,source_schema_name,table_name,source_database_name)
        df.write.format("delta").mode("append").saveAsTable(F"{target_catalog}.bronze.{table_name}")
        time.sleep(20)








       


